# 02 - Baseline model + fixed CV folds

**Goal:** make a simple first score to beat, and create the 5 CV folds that **every** later notebook reuses.

- **Cross-validation (CV):** split train into 5 parts. Train on 4, test on the 5th. Repeat 5 times.
  The average score is our trusted score (more trusted than the public leaderboard).
- **Model:** Logistic regression (a simple straight-line model). Text columns are one-hot encoded
  (one 0/1 column per value), number columns are scaled to similar ranges.
- **Saved:** out-of-fold predictions in `oof/`, test predictions in `preds/`, score in `experiments.csv`.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from config import *
from utils import *
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
train, test, sample = load_data()
folds = get_folds(train)          # creates folds/folds.csv the first time
y = train[TARGET].values
print(pd.Series(folds).value_counts().sort_index())
print("Buy rate per fold:", train.groupby(folds)[TARGET].mean().round(4).tolist())

0    133733
1    133733
2    133733
3    133733
4    133733
Name: count, dtype: int64
Buy rate per fold: [0.1746, 0.1746, 0.1746, 0.1746, 0.1746]


In [3]:
def make_model():
    prep = ColumnTransformer([
        ("num", StandardScaler(), NUM_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
    ])
    return make_pipeline(prep, LogisticRegression(max_iter=1000))

oof, test_pred = np.zeros(len(train)), np.zeros(len(test))
for k in range(N_FOLDS):
    tr, va = folds != k, folds == k
    m = make_model().fit(train.loc[tr, FEATURES], y[tr])
    oof[va] = m.predict_proba(train.loc[va, FEATURES])[:, 1]
    test_pred += m.predict_proba(test[FEATURES])[:, 1] / N_FOLDS
    print(f"fold {k}: AUC = {score(y[va], oof[va])[0]:.5f}")

auc, ll = score(y, oof)
print(f"\nLogistic regression CV  AUC = {auc:.5f}   logloss = {ll:.5f}")

fold 0: AUC = 0.93667


fold 1: AUC = 0.93804


fold 2: AUC = 0.93907


fold 3: AUC = 0.93863


fold 4: AUC = 0.93807



Logistic regression CV  AUC = 0.93809   logloss = 0.23329


In [4]:
save_preds("logreg_v1", oof, test_pred)
log_experiment("logreg_v1", "raw", auc, ll, "one-hot + scaling")
make_submission(test_pred, "logreg_v1", sample)
pd.read_csv(EXPERIMENTS_FILE)

Saved logreg_v1.csv  rows=286571


,model,features,cv_auc,cv_logloss,notes,date
0,logreg_v1,raw,0.938094,0.233295,one-hot + scaling,2026-10-17 15:07
